In [ ]:

# Install missing packages
import subprocess
import sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "scikit-image"])

# Analysis Plan:
# 1. Load the 30-image study set and reconstruct measurement data at 15% sampling
# 2. Implement Blended-HASA with λ_final = λ_static + β * λ_adaptive
# 3. Run reconstructions for β=0.25 and β=0.5
# 4. Load benchmark results from reconstruction_results_15percent_all_methods.csv
# 5. Compare performance metrics against Static-HASA and Adaptive-HASA baselines
# 6. Determine if blended approach meets within-5% criteria on both metrics

import numpy as np
import pandas as pd
import os
from pathlib import Path
from PIL import Image
import pywt
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
from skimage.restoration import denoise_tv_chambolle
from scipy.ndimage import generic_filter
import time
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully")
print(f"Working directory: {os.getcwd()}")


NEWRELIC: 2025-12-03 15:31:28 (75) - New Relic could not start because the newrelic-admin script was called from a Python installation that is different from the Python installation that is currently running. To fix this problem, call the newrelic-admin script from the Python installation that is currently running (details below).

newrelic-admin Python directory: None
current Python directory: '/app/miniconda'
newrelic-admin Python version: None
current Python version: '3.12'


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
clipkit 2.6.1 requires numpy<2.1,>=1.26.0; python_version == "3.12", but you have numpy 2.3.5 which is incompatible.
coredis 5.3.0 requires typing-extensions>=4.13, but you have typing-extensions 4.12.2 which is incompatible.
fcsparser 0.2.8 requires numpy<2,>=1, but you have numpy 2.3.5 which is incompatible.
fhda 1.0.0 requires python-json-logger>=4.0.0, but you have python-json-logger 2.0.7 which is incompatible.
numpydoc 1.5.0 requires sphinx>=4.2, but you have sphinx 3.5.3 which is incompatible.


Libraries loaded successfully
Working directory: /storage/workspace_path


In [ ]:

# Step 1: Define the 30-image study set (15 benign, 15 malignant)
# Based on previous analyses, using a consistent selection

benign_images = [
    'benign (1).png', 'benign (10).png', 'benign (100).png', 'benign (101).png', 
    'benign (102).png', 'benign (103).png', 'benign (104).png', 'benign (105).png',
    'benign (106).png', 'benign (107).png', 'benign (108).png', 'benign (109).png',
    'benign (11).png', 'benign (110).png', 'benign (111).png'
]

malignant_images = [
    'malignant (1).png', 'malignant (10).png', 'malignant (100).png', 
    'malignant (101).png', 'malignant (102).png', 'malignant (103).png',
    'malignant (104).png', 'malignant (105).png', 'malignant (106).png',
    'malignant (107).png', 'malignant (108).png', 'malignant (109).png',
    'malignant (11).png', 'malignant (110).png', 'malignant (111).png'
]

# Create full paths
dataset_path = Path('Dataset_BUSI_with_GT')
image_list = []

for img_name in benign_images:
    image_list.append({
        'class': 'benign',
        'image_path': dataset_path / 'benign' / img_name,
        'mask_path': dataset_path / 'benign' / img_name.replace('.png', '_mask.png')
    })

for img_name in malignant_images:
    image_list.append({
        'class': 'malignant',
        'image_path': dataset_path / 'malignant' / img_name,
        'mask_path': dataset_path / 'malignant' / img_name.replace('.png', '_mask.png')
    })

print(f"Total images in study set: {len(image_list)}")
print(f"Benign: {len(benign_images)}, Malignant: {len(malignant_images)}")


Total images in study set: 30
Benign: 15, Malignant: 15


In [ ]:

# Step 2: Load and preprocess images ONLY (no measurement generation yet)
# We'll generate measurements on-the-fly during reconstruction to save memory

def load_and_preprocess_image(image_path, target_size=(256, 256)):
    """Load image, convert to grayscale, resize, normalize to [0,1]"""
    img = Image.open(image_path).convert('L')
    img = img.resize(target_size, Image.BILINEAR)
    img_array = np.array(img, dtype=np.float64) / 255.0
    return img_array

def load_mask(mask_path, target_size=(256, 256)):
    """Load mask and resize"""
    mask = Image.open(mask_path).convert('L')
    mask = mask.resize(target_size, Image.NEAREST)
    mask_array = np.array(mask, dtype=bool)
    return mask_array

# Load all images and masks (lightweight operation)
print("Loading images and masks...")
data_records = []

for idx, item in enumerate(image_list):
    img = load_and_preprocess_image(item['image_path'])
    mask = load_mask(item['mask_path'])
    
    data_records.append({
        'index': idx,
        'class': item['class'],
        'image_name': item['image_path'].name,
        'original_image': img,
        'mask': mask
    })

print(f"Completed loading {len(data_records)} images")
print(f"Image shape: {data_records[0]['original_image'].shape}")
